# 🧩 Laboratorio de Clustering: los grupos que ya existen entre sus clientes

**Diplomado en Inteligencia Artificial · UNAC · Clase 04 · Plantilla del aula**

### 🎯 Qué vamos a hacer

- Tomamos una base de **600 clientes de un supermercado** y le pedimos a la máquina que, sin que nadie le diga nada, encuentre los grupos naturales de compradores.
- Le **escondemos a propósito** la columna que ya dice a qué segmento pertenece cada cliente. El modelo trabaja a ciegas.
- Al final **destapamos la respuesta** y calificamos al modelo: ¿llegó solo a los mismos grupos que la gerencia ya conocía?
- Le ponemos nombre de negocio a cada grupo y nos llevamos un Excel listo para mostrárselo al jefe.

### 🧭 Cómo se usa

**Usted no tiene que escribir una sola línea de código.**

1. Corra las celdas una por una con el botón ▶ que aparece a la izquierda de cada bloque gris, o todas de una con **Entorno de ejecución → Ejecutar todas**.
2. Lo **ÚNICO** que se cambia en este notebook está en la celda del **PANEL DE CONTROL**. Todo lo demás se deja quieto.
3. Hay una celda que le va a pedir **reiniciar la sesión**. Es normal, está explicada y no se salta.

⏱️ Tiempo estimado: 20 minutos, de los cuales 3 son esperando la instalación.

## 0️⃣ ¿Qué hay que instalar y por qué toca reiniciar?

Google Colab trae Python listo, pero no trae la librería que vamos a usar: **PyCaret**, que es la que hace todo el trabajo pesado en tres o cuatro renglones.

Al instalarla, Colab tiene que cambiar de versión unos paquetes que ya tenía cargados en memoria. Por eso, cuando termine, **hay que reiniciar la sesión**: es como apagar y prender el computador para que reconozca algo que se acabó de instalar.

Ojo con el nombre, que es la trampa del año: se instala **`pycaret-core`**, que es la versión que la comunidad mantiene al día. La de toda la vida (`pycaret`, a secas) quedó abandonada y ya no funciona con el Python que corre Colab hoy.

**Qué mirar en el resultado:** que la celda termine (deja de girar el círculo). Los avisos amarillos que digan *incompatible* son normales y se resuelven al reiniciar.

In [ ]:
# La UNICA instalacion de este notebook. Tarda entre 2 y 4 minutos: es normal, dejela correr.
# OJO: es "pycaret-core", NO "pycaret". El paquete viejo ya no sirve en el Colab de hoy.
!pip install pycaret-core -q
print("Instalacion terminada. Ahora SI hay que reiniciar la sesion: lea la celda de abajo.")

---

# 🔴 PARE AQUÍ: hay que reiniciar la sesión

No siga de largo. Haga esto, en este orden:

1. En el menú de arriba: **Entorno de ejecución → Reiniciar sesión**
   *(si su Colab está en inglés: **Runtime → Restart session**)*
2. Cuando pregunte si está seguro, diga **Sí**.
3. Espere unos 10 segundos a que Colab vuelva.
4. Siga **desde la celda de abajo**, y **NO vuelva a correr la celda del `pip install`**.

> Si la vuelve a correr no se rompe nada, pero pierde otros tres minutos esperando.

**Por qué:** los paquetes viejos siguen vivos en la memoria de Colab hasta que se reinicia. Sin reiniciar, la siguiente celda va a decir que no encuentra PyCaret aunque sí esté instalado.

**Qué mirar en el resultado:** al correr la celda de abajo debe aparecer `Listo: pycaret ... quedo cargado`.

---

In [ ]:
# Semaforo: confirma que el reinicio quedo bien hecho antes de seguir.
try:
    import pycaret
    print("Listo: pycaret", pycaret.__version__, "quedo cargado. Puede seguir tranquilo.")
except ModuleNotFoundError:
    print("PARE. No encuentro pycaret. Solo hay dos causas posibles:")
    print("1) No reinicio la sesion -> Entorno de ejecucion > Reiniciar sesion, y vuelva a correr ESTA celda.")
    print("2) La instalacion no alcanzo a terminar -> corra otra vez la celda del pip install y luego reinicie.")

## 1️⃣ ¿Qué datos tenemos?

Vamos a trabajar con la base de **600 clientes de un supermercado**: cuánto factura cada uno al mes, cuántas veces viene, de cuánto es su compra promedio, hace cuánto es cliente y qué tanto descuento usa.

Los datos los fabrica el notebook en el momento —no hay nada que descargar—, pero están hechos para parecerse a lo que sale de un sistema de caja de verdad: con ruido, con clientes raros y con casos de frontera.

La última columna, `segmento_conocido`, es el segmento que la gerencia **ya tiene identificado**. Téngala presente: es la respuesta del examen y en un minuto la vamos a esconder.

Un detalle que está puesto a propósito: cerca del **13% de estos clientes se porta como los del segmento de al lado**. Por eso el modelo no va a acertar al 100%, y así es exactamente en la vida real: siempre hay clientes que no se dejan encasillar.

**Qué mirar en el resultado:** que diga **600 filas y 6 columnas**, y que en la tabla aparezcan segmentos distintos (VIP, Ocasional, Caza-ofertas).

In [ ]:
# Esta celda es la FABRICA DE DATOS: no hay que leerla, solo correrla.
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)   # semilla: a todo el salon le sale lo mismo
cuantos = {"VIP": 150, "Ocasional": 250, "Caza-ofertas": 200}
# como se comporta en promedio cada segmento: (visitas, ticket, antiguedad, descuento)
centro = {"VIP": (12, 180000, 48, 4), "Ocasional": (3, 65000, 14, 6), "Caza-ofertas": (8, 55000, 26, 28)}
ruido = {"VIP": (3, 30000, 14, 2), "Ocasional": (1.2, 15000, 7, 3), "Caza-ofertas": (2.5, 12000, 10, 6)}

etiqueta = np.repeat(list(cuantos.keys()), list(cuantos.values()))
# El 13% son clientes "de frontera": se portan como los de otro segmento. La vida real no es perfecta.
porta_como = etiqueta.copy()
frontera = rng.random(len(etiqueta)) < 0.13
porta_como[frontera] = rng.choice(list(cuantos.keys()), frontera.sum())

def columna(i):
    return np.array([rng.normal(centro[s][i], ruido[s][i]) for s in porta_como])

clientes = pd.DataFrame({
    "visitas_mes": np.clip(columna(0), 1, None).round(0),
    "ticket_promedio": np.clip(columna(1), 8000, None).round(0),
    "antiguedad_meses": np.clip(columna(2), 1, None).round(0),
    "porcentaje_descuento": np.clip(columna(3), 0, 60).round(1),
    "segmento_conocido": etiqueta,
})
clientes.insert(0, "facturacion_mes",
                (clientes["visitas_mes"] * clientes["ticket_promedio"] * rng.normal(1, 0.06, len(clientes))).round(0))
clientes = clientes.sample(frac=1, random_state=42).reset_index(drop=True)
print("Tabla de clientes:", clientes.shape[0], "filas y", clientes.shape[1], "columnas")
clientes.head(8)

## 2️⃣ El PANEL DE CONTROL

> # 🎛️ AQUÍ Y SOLO AQUÍ SE CAMBIAN COSAS

Estas son las cinco perillas del laboratorio. Muévalas con los controles que aparecen a la derecha de la celda; no hace falta tocar el texto del código.

| Perilla | Para qué sirve |
|---|---|
| `NUMERO_DE_GRUPOS` | En cuántos grupos quiere partir la clientela. Empiece en **3**. |
| `ALGORITMO` | Cuál de las tres formas de agrupar se usa. Empiece en **kmeans**. |
| `COLUMNAS_A_ESCONDER` | Las columnas que el modelo **no** puede ver. Aquí va la respuesta del examen. |
| `NOMBRE_DEL_EXCEL` | Cómo se va a llamar el archivo que se descarga al final. |
| `MOSTRAR_GRAFICAS` | Apáguelo solo si quiere que corra más rápido. |

Si quiere esconder más de una columna, sepárelas con coma: `segmento_conocido, ciudad`.

**Qué mirar en el resultado:** que el mensaje repita exactamente los valores que usted escogió.

In [ ]:
# ===================== PANEL DE CONTROL: AQUI Y SOLO AQUI SE CAMBIAN COSAS =====================
NUMERO_DE_GRUPOS = 3  #@param {type:"slider", min:2, max:8, step:1}
ALGORITMO = "kmeans"  #@param ["kmeans", "hclust", "birch"]
COLUMNAS_A_ESCONDER = "segmento_conocido"  #@param {type:"string"}
NOMBRE_DEL_EXCEL = "clientes_segmentados.xlsx"  #@param {type:"string"}
MOSTRAR_GRAFICAS = True  #@param {type:"boolean"}
# ==============================================================================================

COLUMNAS_A_ESCONDER = [c.strip() for c in COLUMNAS_A_ESCONDER.split(",") if c.strip()]
print("Grupos a buscar:", NUMERO_DE_GRUPOS, "  |  Algoritmo:", ALGORITMO)
print("Columnas que el modelo NO va a ver:", COLUMNAS_A_ESCONDER)
print("Excel de salida:", NOMBRE_DEL_EXCEL)

## 3️⃣ ¿Qué le escondemos al modelo y para qué?

Esto es el corazón del ejercicio y es la idea que hay que llevarse de la clase.

Ya sabemos a qué segmento pertenece cada cliente. Vamos a **taparle esa columna al modelo** y dejarlo agrupar a ciegas, solo con los números de comportamiento. Al final destapamos la respuesta y vemos si llegó por su cuenta al mismo sitio.

Eso es **aprendizaje no supervisado**: nadie le dice cuál es la respuesta correcta, el modelo encuentra la estructura que ya estaba en los datos. En su empresa casi nunca va a tener la respuesta a mano; por eso vale la pena comprobar aquí, donde sí la tenemos, que el método funciona.

De paso hacemos la única limpieza de la plantilla: botar las filas incompletas. Nada más.

**Qué mirar en el resultado:** que la lista de columnas que el modelo va a ver **NO** incluya `segmento_conocido`.

In [ ]:
clientes = clientes.dropna(axis=0)                             # unica limpieza: fuera las filas incompletas
datos_para_agrupar = clientes.drop(COLUMNAS_A_ESCONDER, axis=1)  # aqui se esconde la respuesta

print("El modelo va a ver SOLO estas columnas:", list(datos_para_agrupar.columns))
print("Y esta no la va a ver:", COLUMNAS_A_ESCONDER, "-> la destapamos al final")
print("Filas que entran al modelo:", len(datos_para_agrupar))
datos_para_agrupar.head()

## 4️⃣ ¿Por qué hay que poner todo en la misma escala?

Las columnas vienen en unidades muy distintas: la facturación está en millones de pesos y las visitas son un número de un dígito. Si las dejamos así, la facturación manda sobre todas las demás solo por ser un número más grande, no por ser más importante.

`normalize=True` arregla eso en una palabra: pone todas las columnas en la misma escala antes de agrupar. Es lo único de "preparación de datos" que hace esta plantilla, y es suficiente.

`session_id=42` es la semilla, para que a usted y al de al lado les salga exactamente el mismo resultado. `use_gpu=False` porque aquí no hace falta tarjeta gráfica y pedirla solo trae problemas. Y `numeric_features` es un aviso a PyCaret de que todas esas columnas son números de verdad y no códigos ni categorías.

**Qué mirar en el resultado:** el mensaje de que los datos quedaron escalados. Aquí todavía no hay grupos.

In [ ]:
from pycaret.clustering import setup, create_model, assign_model, plot_model, evaluate_model

columnas_numericas = list(datos_para_agrupar.select_dtypes("number").columns)

experimento = setup(
    data=datos_para_agrupar,
    numeric_features=columnas_numericas,  # aviso: todo esto son numeros, no categorias
    normalize=True,      # todas las columnas a la misma escala
    session_id=42,       # semilla: resultado igual para todo el salon
    use_gpu=False,       # sin GPU: no aporta nada en agrupamiento
    verbose=False,       # sin la tabla larga de configuracion
)
print("Datos escalados y listos:", len(datos_para_agrupar), "clientes y",
      len(datos_para_agrupar.columns), "columnas en la misma escala.")

## 5️⃣ Que la máquina encuentre los grupos

Un solo renglón. `create_model` toma la tabla escalada y arma los grupos que le pedimos.

Tres formas de agrupar sirven para este caso —`kmeans`, `hclust` y `birch`— y el criterio del gerente es simple: se prueban las tres y se queda con la que separe mejor. No hay que entender la matemática de ninguna para poder decidir.

**Qué mirar en el resultado:** la fila **Silhouette**. Va de -1 a 1 y mide qué tan separados quedaron los grupos: por encima de **0,5** ya es una separación clara y utilizable. Anote el número, que lo vamos a comparar.

In [ ]:
modelo = create_model(ALGORITMO, num_clusters=NUMERO_DE_GRUPOS)
print()
print("Esto es lo que quedo armado por dentro (no hay que entenderlo, solo verlo):")
print(modelo)

## 6️⃣ ¿Y cómo sé que 3 era el número correcto de grupos?

Dos gráficas ayudan a decidir:

- **Codo (elbow):** a medida que se agregan grupos, el error baja. El "codo" es el punto donde la mejora deja de valer la pena; ahí está el número razonable.
- **Silueta (silhouette):** mide qué tan cómodo está cada cliente en el grupo que le tocó. Barras anchas y parejas significan buenos grupos.

Y ahora el criterio que sí es de gerencia y no de matemática: **a veces el número de grupos lo fija la pregunta del negocio, no la curva.** Si usted solo puede montar dos campañas al mes, son dos grupos, así el codo diga cuatro.

Estas dos gráficas están hechas para `kmeans`. Con `hclust` a veces no se pueden dibujar, y no es grave.

**Qué mirar en el resultado:** en el codo, dónde se dobla la línea (la línea punteada lo señala). En la silueta, que ningún grupo se quede muy a la izquierda de la línea del promedio.

In [ ]:
if MOSTRAR_GRAFICAS:
    for grafica, nombre in [("elbow", "curva de codo"), ("silhouette", "grafica de silueta")]:
        try:
            plot_model(modelo, plot=grafica)
        except Exception as detalle:
            print("No se pudo dibujar la", nombre, "en este entorno. Siga con la clase, no es grave.")
            print("Detalle tecnico:", detalle)
else:
    print("Graficas apagadas en el PANEL DE CONTROL.")

## 7️⃣ ¿En qué grupo cayó cada cliente?

`assign_model` devuelve la misma tabla con una columna nueva al final: **Cluster**. Ahí dice, cliente por cliente, en qué grupo lo puso el modelo.

Y aquí mismo hacemos el destape: le volvemos a pegar la columna `segmento_conocido` que le habíamos escondido. El modelo ya decidió, así que ya no puede hacer trampa.

**Qué mirar en el resultado:** las dos últimas columnas de la tabla, una al lado de la otra. Compárelas fila por fila y vaya haciéndose una idea antes de ver el puntaje.

In [ ]:
resultado = assign_model(modelo)

# Destape: le devolvemos la columna que le habiamos escondido, ya con el modelo decidido
for columna_escondida in COLUMNAS_A_ESCONDER:
    resultado[columna_escondida] = clientes.loc[resultado.index, columna_escondida]

print("Clientes clasificados:", len(resultado))
print("Cuantos quedaron en cada grupo:")
print(resultado["Cluster"].value_counts().sort_index().to_string())
resultado.head(8)

## 8️⃣ ¿Cómo se ven esos grupos en un mapa?

Un cliente, un punto. El color es el grupo que le asignó el modelo. Si los colores forman manchas separadas, la segmentación sirve; si están revueltos, no sirve.

La primera gráfica usa dos columnas de negocio que usted le puede explicar a cualquiera: cuánto factura y cuánto descuento usa (si más adelante cambia los datos por los de su empresa, toma solas las dos columnas que corresponda). La segunda la dibuja PyCaret, y lo que hace es aplastar las cinco columnas en un solo plano para verlas todas a la vez.

**Qué mirar en el resultado:** que se distingan **tres manchas** de colores distintos, y sobre todo la esquina de mucho descuento con poca facturación: ahí viven los caza-ofertas.

In [ ]:
import matplotlib.pyplot as plt

# Para el mapa usamos dos columnas: la primera y la ultima de las numericas.
# Asi la grafica sigue saliendo cuando usted cambie los datos por los de su empresa.
eje_x, eje_y = columnas_numericas[0], columnas_numericas[-1]
escala = 1000 if resultado[eje_x].max() > 100000 else 1

for grupo, bloque in resultado.groupby("Cluster"):
    plt.scatter(bloque[eje_x] / escala, bloque[eje_y], alpha=0.65, label=grupo)
plt.title("Grupos de clientes que el modelo encontro por su cuenta")
plt.xlabel(eje_x.replace("_", " ") + (" (en miles)" if escala == 1000 else ""))
plt.ylabel(eje_y.replace("_", " "))
plt.legend(title="Grupo")
plt.show()

# El mismo mapa, pero dibujado por PyCaret con las cinco columnas a la vez
if MOSTRAR_GRAFICAS:
    try:
        plot_model(modelo, plot="cluster")
    except Exception as detalle:
        print("El mapa interactivo de PyCaret no cargo en este entorno; con el de arriba basta.")
        print("Detalle tecnico:", detalle)

## 9️⃣ El momento de la verdad: ¿le acertó el modelo?

Antes de leer los números, tenga claro qué es cada cosa: **las filas** son los tres segmentos que la gerencia ya conocía (VIP, Ocasional, Caza-ofertas) y **las columnas** son los grupos que el modelo inventó por su cuenta, sin verlos nunca.

El modelo no sabe cómo se llaman los segmentos, así que no espere una diagonal perfecta: el Cluster 0 puede ser justo el que la gerencia llama VIP. Lo que importa es que **cada fila se concentre en una sola columna**.

**Qué mirar en el resultado:** el porcentaje de coincidencia. Entre **85% y 92%** es lo esperable, y es una buena noticia: significa que los grupos eran reales y estaban ahí, en los datos, desde antes. Si saliera 100% habría que desconfiar; la vida real no es tan limpia.

In [ ]:
from IPython.display import display

if not COLUMNAS_A_ESCONDER:
    print("No escondio ninguna columna, entonces no hay respuesta con que comparar.")
    print("Vuelva al PANEL DE CONTROL y escriba: segmento_conocido")
else:
    etiqueta_real = COLUMNAS_A_ESCONDER[0]
    tabla = pd.crosstab(resultado[etiqueta_real], resultado["Cluster"])
    # Coincidencia: a cada grupo del modelo le asignamos el segmento real que mas aparece dentro
    coincidencia = tabla.max(axis=0).sum() / tabla.values.sum() * 100
    print("Coincidencia entre los grupos que el modelo armo a ciegas y los segmentos reales:",
          str(round(coincidencia, 1)) + "%")
    display(tabla)

## 🔟 ¿Cómo es cada grupo por dentro?

Ya sabemos que hay tres grupos y que son de fiar. Ahora hay que saber **quiénes son**, y para eso se mira el promedio de cada columna dentro de cada grupo. Esta tabla es la que se lleva al comité.

**Qué mirar en el resultado:** en cada fila, la columna donde el número se sale del promedio general. Un grupo con facturación alta y descuento bajo es un cliente fiel; uno con descuento alto y ticket bajo viene solo por la promoción.

In [ ]:
perfil = resultado.groupby("Cluster").mean(numeric_only=True).round(1)
print("Promedio de cada columna dentro de cada grupo:")
perfil

## 1️⃣1️⃣ ✍️ El bautizo: póngale nombre de negocio a cada grupo

Este es el entregable de verdad. Un número de cluster no le sirve a nadie; un nombre sí.

Mire la tabla de promedios de arriba y **escriba aquí mismo** —doble clic sobre este texto para editarlo— cómo se llama cada grupo en el idioma de su empresa:

| Grupo del modelo | Nombre de negocio | Qué haría con ellos |
|---|---|---|
| Cluster 0 | *(escriba aquí)* | *(escriba aquí)* |
| Cluster 1 | *(escriba aquí)* | *(escriba aquí)* |
| Cluster 2 | *(escriba aquí)* | *(escriba aquí)* |

Así se ve bien hecho:

- *"Cluster 0 = clientes de alto valor y visita frecuente → programa de fidelización, y ni un peso de descuento."*
- *"Cluster 2 = compradores de promoción → sirven para mover inventario que está por vencerse."*

Cuando termine de escribir, oprima **Shift + Enter** para que el texto se vea formateado.

## 1️⃣2️⃣ ¿Cómo se lo muestro a mi jefe?

Un Excel con los 600 clientes y el grupo de cada uno. Se descarga a su computador y se abre con doble clic, sin conversiones ni pasos raros.

**Qué mirar en el resultado:** que el navegador empiece la descarga. Si no arranca sola, abra el archivo desde el icono de carpeta 📁 de la barra izquierda de Colab.

In [ ]:
resultado.to_excel(NOMBRE_DEL_EXCEL, index=False)
print("Archivo creado:", NOMBRE_DEL_EXCEL, "con", len(resultado), "clientes clasificados.")

try:
    from google.colab import files
    files.download(NOMBRE_DEL_EXCEL)
except Exception as detalle:
    print("La descarga automatica no arranco. Busque el archivo en el icono de carpeta de la izquierda.")
    print("Detalle tecnico:", detalle)

## 🏁 Ahora con SUS datos

La plantilla es la misma. Lo único que cambia es de dónde salen las filas.

**Paso 1 — arme el archivo.** Un Excel o un CSV donde **cada fila sea un cliente** (o un proveedor, un paciente, una obra, un contrato) y **cada columna sea un número** que describa su comportamiento. Nada de texto libre, nada de celdas combinadas, nada de fila de totales al final. Con 4 a 8 columnas numéricas basta, y mínimo unas 100 filas.

**Paso 2 — súbalo a Colab.** En la barra izquierda, icono de carpeta 📁 → botón de subir archivo, o arrástrelo ahí. Queda disponible mientras la sesión esté abierta; si Colab se reinicia, hay que volverlo a subir.

**Paso 3 — reemplace la celda que fabrica los datos.** En la sección 1️⃣, donde el notebook inventa los 600 clientes con `pd.DataFrame(...)`, va en su lugar **una sola línea** que lee su archivo: `pd.read_excel` con el nombre del archivo que subió, o `pd.read_csv` si es CSV. Si prefiere subirlo con un botón desde dentro del notebook, `files.upload()` de `google.colab` también sirve. Pídale a su Claude que le escriba ese renglón: es más rápido que buscarlo.

**Paso 4 — ajuste el PANEL DE CONTROL.** En `COLUMNAS_A_ESCONDER` van las columnas que el modelo no debe ver: nombres, cédulas, ciudad, y la etiqueta si usted ya tiene una. En `NUMERO_DE_GRUPOS`, cuántos grupos quiere.

Y ya. El resto del notebook no se toca.

> ⚠️ **Antes de subir nada:** si el archivo tiene datos de clientes reales, quítele nombres, cédulas, teléfonos y correos. Para agrupar no se necesitan, y Colab es un computador de Google, no el suyo.

### 🤖 Pregúntele a su Claude

No hay que aprenderse nada de esto: se le pide. Copie uno de estos y péguelo en su Claude.

**Para adaptar la plantilla a su empresa:**

> "Adapta este notebook de clustering con PyCaret a mis datos. Mi archivo se llama `______.xlsx` y tiene estas columnas: `______`, `______`, `______`, `______`. Dime exactamente qué celda cambio y por qué texto la reemplazo, sin tocar nada más del notebook."

**Para entender el resultado que le salió:**

> "Corrí un clustering y me quedaron 3 grupos con estos promedios: [pegue aquí la tabla de promedios]. Ayúdame a ponerle a cada grupo un nombre de negocio y dime qué acción comercial concreta haría con cada uno."

**Para decidir el número de grupos:**

> "Mi silhouette con 3 grupos fue ____ y con 4 fue ____. Somos [su tipo de negocio] y podemos manejar máximo ____ campañas distintas al mes. ¿Con cuántos grupos me quedo y cómo lo justifico ante la gerencia?"

## 🔍 Para curiosos: el panel de PyCaret

Opcional, y solo si va bien de tiempo. `evaluate_model` abre un panel con botones para ver todas las gráficas del modelo desde un mismo sitio. A veces no carga bien en Colab; si no aparece, no se preocupe, ya vimos lo importante.

**Qué mirar en el resultado:** los botones de arriba del panel. El de **Distribution** es el más útil: muestra cómo se reparte cada columna dentro de cada grupo.

In [ ]:
try:
    evaluate_model(modelo)
except Exception as detalle:
    print("El panel interactivo no cargo en este Colab. No pasa nada: las graficas importantes ya salieron.")
    print("Detalle tecnico:", detalle)

---

**Diplomado en IA · UNAC · Clase 04 · Si algo falla, copie el error completo y péguelo a su Claude: él sabe de este notebook.**